# Monte Carlo: estimating $\pi$ with random darts

*others · beginner* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/others/monte_carlo_pi)

Throw random darts at a unit square and count how many land inside the quarter circle: the fraction tends to $\pi/4$. Many independent experiments run side by side in one $(B, N)$ tensor, which shows the defining law of Monte Carlo, an error that shrinks only like $1/\sqrt N$. Buffon's needle gives a second, 18th-century estimate.

**Vector form**

$$\frac{\pi}{4} = \int_0^1\!\!\int_0^1 [x^2 + y^2 \le 1]\,dx\,dy \approx \frac1N\sum_{n=1}^N h_n,\qquad \sigma_N = \sqrt{\frac{\pi(4-\pi)}{N}}$$

**Index (tensor) form**

$$h_{bn} = [x_{bni}\,x_{bni} \le 1],\qquad \hat\pi_{bn} = \frac{4}{n}\sum_{m \le n} h_{bm},\qquad \sigma_n = \operatorname{std}_b\,\hat\pi_{bn}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $x_{bni}$ | `torch.rand(B, N, 2)` | (B, N, 2) | dart $n$ of experiment $b$, coordinate $i$, uniform in $[0, 1)$ |
| $h_{bn}$ | `einsum('bni,bni->bn', x, x) <= 1` | (B, N) | repeated $i$ ⇒ summed: squared distance from the corner; 1 if the dart is inside the quarter circle |
| $\hat\pi_{bn}$ | `running_estimate(h, 4.0)` | (B, N) | cumulative sum divided by $n$: the estimate after every dart, for every experiment |
| $\sigma_n$ | `est.std(dim=0)` | (N,) | spread across experiments; theory $\sqrt{\pi(4-\pi)/n}$ from the binomial variance |
| $[y \le \tfrac{\ell}{2}\sin\varphi]$ | `needle_crosses(y, phi, length)` | (B, N) | Buffon's needle: crosses a line with probability $2\ell/(\pi d)$, so $\hat\pi = 2\ell N/(d\,\text{crossings})$ |

Every estimate is right on average but noisy, and the noise falls only as $1/\sqrt N$: a hundred times more darts for one more correct digit. That rate does not depend on the dimension, which is why Monte Carlo is the method of choice for integrals over many variables, from neutron transport and the Ising model to option pricing and photon paths in medical dosimetry. The grey curves are single experiments: each wanders, but they all stay in the band set by $\sigma_N$. Fixed seeds make every run reproducible; try more experiments to see the spread match the theory more closely.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/others/monte_carlo_pi/main.py
"""Main loop: parameters → tensors → solve → plot."""

import math
from pathlib import Path

import torch

from problems._common.runtime import cli, device

from . import plot, solve, tensors
from .equation import standard_error


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    b, n = params["n_experiments"], params["n_darts"]
    x = tensors.darts(b, n, 0, dev)  # (B, N, 2)
    est = solve.darts_estimate(x)  # (B, N)
    sd = solve.spread(est)  # (N,)
    ns = torch.arange(1, n + 1, dtype=torch.float64, device=dev)

    print(f"{b} experiments × {n} darts")
    for k in (100, 1000, 10000, n):
        if k <= n:
            print(
                f"N = {k:7d}: mean π̂ = {est[:, k - 1].mean():.5f}   spread {sd[k - 1]:.5f}   "
                f"theory √(π(4−π)/N) = {standard_error(ns[k - 1]):.5f}"
            )
    final = est[:, -1]
    dev_from_pi = (final - math.pi).abs()
    print(f"best experiment {final[dev_from_pi.argmin()]:.6f}, worst {final[dev_from_pi.argmax()]:.6f}")

    y, phi = tensors.needles(b, n, 0, dev)
    needle = solve.needle_estimate(y, phi, params["needle_length"])  # (B,)
    print(f"Buffon's needle (ℓ/d = {params['needle_length']}): π̂ = {needle.mean():.5f} ± {needle.std():.5f}")
    print("4× the darts gives only 2× the accuracy: the error falls as 1/√N in every dimension")
    return plot.animate(x, est, sd, out_dir / "monte_carlo_pi.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/others/monte_carlo_pi/equation.py
r"""Random numbers → an integral.

Monte Carlo replaces an integral by an average over random samples. A dart x uniform in the unit square lands
inside the quarter circle x_i x_i ≤ 1 with probability equal to its area, π/4, so

    \hat\pi_N = \frac{4}{N}\sum_{n=1}^{N} h_n,      h_n = [x_{ni} x_{ni} \le 1]      (i summed).

Each h_n is a coin flip with p = π/4, so the estimate is unbiased and its standard error is

    \sigma_N = 4\sqrt{p(1 - p)/N} = \sqrt{\pi(4 - \pi)/N} \propto 1/\sqrt N.

Buffon's needle (1777): a needle of length ℓ ≤ d dropped on lines a distance d apart crosses one with probability
2ℓ/(πd); its centre lies a distance y from the nearest line and it crosses when y ≤ (ℓ/2) sin φ.
"""

import math

import torch


def hits(x: torch.Tensor) -> torch.Tensor:
    r"""h_{bn} = [x_{bni} x_{bni} ≤ 1]   (i summed).   (B, N, 2) → (B, N)"""
    return (torch.einsum("bni,bni->bn", x, x) <= 1.0).double()


def running_estimate(h: torch.Tensor, scale: float) -> torch.Tensor:
    r"""\hat\pi_{bn} = scale · (h_{b1} + … + h_{bn}) / n for every prefix length n.   (B, N) → (B, N)"""
    n = torch.arange(1, h.shape[-1] + 1, dtype=h.dtype, device=h.device)
    return scale * torch.cumsum(h, dim=-1) / n


def standard_error(n: torch.Tensor) -> torch.Tensor:
    r"""σ_N = √(π(4 − π)/N) for the dart estimate."""
    return torch.sqrt(math.pi * (4 - math.pi) / n)


def needle_crosses(y: torch.Tensor, phi: torch.Tensor, length: float) -> torch.Tensor:
    r"""[y ≤ (ℓ/2) sin φ] with line spacing d = 1.   (B, N) → (B, N)"""
    return (y <= 0.5 * length * torch.sin(phi)).double()


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/others/monte_carlo_pi/tensors.py
"""Build the concrete tensors: random darts in the unit square and random needles on a ruled floor."""

import math

import torch


def darts(n_experiments: int, n_darts: int, seed: int, device: torch.device) -> torch.Tensor:
    """x_{bni}: dart n of experiment b, uniform in the unit square [0, 1)²: shape (B, N, 2)."""
    gen = torch.Generator().manual_seed(seed)
    return torch.rand(n_experiments, n_darts, 2, generator=gen, dtype=torch.float64).to(device)


def needles(n_experiments: int, n_needles: int, seed: int, device: torch.device) -> tuple[torch.Tensor, torch.Tensor]:
    """Needle centres' distance y to the nearest line, uniform in [0, d/2) with d = 1, and angles φ in [0, π/2):
    two tensors of shape (B, N)."""
    gen = torch.Generator().manual_seed(seed + 1)
    y = 0.5 * torch.rand(n_experiments, n_needles, generator=gen, dtype=torch.float64)
    phi = 0.5 * math.pi * torch.rand(n_experiments, n_needles, generator=gen, dtype=torch.float64)
    return y.to(device), phi.to(device)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/others/monte_carlo_pi/solve.py
"""Estimate π both ways for every experiment and measure how the error shrinks with N."""

import torch

from .equation import hits, needle_crosses, running_estimate


def darts_estimate(x: torch.Tensor) -> torch.Tensor:
    """Running π estimate from the darts, shape (B, N)."""
    return running_estimate(hits(x), 4.0)


def needle_estimate(y: torch.Tensor, phi: torch.Tensor, length: float) -> torch.Tensor:
    """π ≈ 2ℓN / (d · crossings), shape (B,); inf if no needle crossed."""
    crossings = needle_crosses(y, phi, length).sum(dim=-1)
    return 2 * length * y.shape[-1] / crossings


def spread(estimates: torch.Tensor) -> torch.Tensor:
    """Standard deviation across the B experiments at every N: (B, N) → (N,)."""
    return estimates.std(dim=0)


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/others/monte_carlo_pi/plot.py
"""Darts landing in the square next to every experiment's error shrinking like 1/√N."""

import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import frame_indices, save_gif


def animate(x: torch.Tensor, est: torch.Tensor, sd: torch.Tensor, path: Path) -> Path:
    """x (B, N, 2), est (B, N), sd (N,)."""
    pts = x[0].cpu().numpy()
    inside = (pts**2).sum(-1) <= 1
    steps = np.unique(np.geomspace(10, est.shape[1], 80).astype(int))  # log-spaced frame sizes
    curve = np.unique(np.geomspace(1, est.shape[1], 400).astype(int))  # log-spaced points for the error curves
    n_all = curve
    err = np.abs(est.cpu().numpy()[:, curve - 1] - math.pi)
    sd = sd[torch.as_tensor(curve - 1)]
    shown = min(len(pts), 4000)  # draw at most this many darts

    fig, (ax_d, ax_e) = plt.subplots(1, 2, figsize=(9.5, 4.3), gridspec_kw={"width_ratios": [1, 1.3]})
    ax_d.set_xlim(0, 1)
    ax_d.set_ylim(0, 1)
    ax_d.set_aspect("equal")
    t = np.linspace(0, math.pi / 2, 100)
    ax_d.plot(np.cos(t), np.sin(t), color="#374151", lw=1.5)
    in_dots = ax_d.scatter([], [], s=3, color="#2563eb")
    out_dots = ax_d.scatter([], [], s=3, color="#f59e0b")
    title = ax_d.set_title("", fontsize=10, family="monospace")

    for b in range(err.shape[0]):
        ax_e.loglog(n_all, np.maximum(err[b], 1e-7), color="#9ca3af", lw=0.6, alpha=0.6)
    ax_e.loglog(n_all, sd.cpu().numpy(), color="#2563eb", lw=2, label="spread over experiments")
    ax_e.loglog(n_all, np.sqrt(math.pi * (4 - math.pi) / n_all), color="#f59e0b", ls="--", lw=1.5, label="√(π(4−π)/N)")
    marker = ax_e.axvline(10, color="#dc2626", lw=1)
    ax_e.set_ylim(1e-5, 2)
    ax_e.set_xlabel("darts N")
    ax_e.set_ylabel("|π̂ − π|")
    ax_e.set_title("error of each experiment (grey)", fontsize=10)
    ax_e.legend(fontsize=8, frameon=False, loc="lower left")
    ax_e.grid(True, which="both", alpha=0.2)

    def update(k: int):
        n = steps[k]
        m = min(n, shown)
        in_dots.set_offsets(pts[:m][inside[:m]])
        out_dots.set_offsets(pts[:m][~inside[:m]])
        title.set_text(f"N = {n:7d}   π̂ = {est[0, n - 1].item():.5f}")
        marker.set_xdata([n, n])

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(steps), 80), path, fps=10)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'n_darts': 100000,
    'n_experiments': 16,
    'needle_length': 0.8
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.others.monte_carlo_pi")]:
    del sys.modules[name]
main = importlib.import_module("problems.others.monte_carlo_pi.main")
problem_dir = Path("problems/others/monte_carlo_pi")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
